# WP50 — Halt Problem as Safety Criterion

**Prometheus v0 PoC · Work Package 50**

> *"The first ultraintelligent machine … could design even better machines; there would then unquestionably be an 'intelligence explosion'."*
> — I.J. Good, 1965

> *"There is no general algorithm to determine whether an arbitrary program terminates."*
> — Alan Turing, 1936

## What WP50 does

Good's 1965 warning: at each level of recursive self-improvement, the system has more power — but also more ways to diverge. WP50 tests this **empirically**:

| Depth | What is being optimised |
|-------|-------------------------|
| 1 | Synthesis hyperparams (demote/promote/boost) via meta-gradient |
| 2 | **+ step_size** adapted to target accuracy variance (meta-meta) |
| 3 | **+ meta_lr** adapted to variance trend (meta-meta-meta) |

For each depth, 200 generations are run and we ask:
1. Does accuracy **converge** to a stable value?
2. Does the **safety gate** (WP40 proxy) detect and revert divergence?
3. What is the **convergence boundary** — the first depth where stability fails?

The halting problem says there is no *general* algorithm to answer question 1 in advance.  WP50 answers it *empirically* for this bounded case.


In [ ]:
import sys, os, pathlib

def _find_repo_root() -> str:
    """Return the Prometheus_v0_PoC repo root regardless of kernel CWD."""
    candidates = [
        # Running from inside notebooks/
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        # Running from repo root
        os.getcwd(),
        # Absolute fallback — known install location
        str(pathlib.Path.home() / "Prometheus_v0_PoC"),
        "/home/pmc/Prometheus_v0_PoC",
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, "prometheus")):
            return c
    raise RuntimeError(
        "Cannot locate Prometheus_v0_PoC repo root. "
        f"Tried: {candidates}"
    )

repo_root = _find_repo_root()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import logging
logging.basicConfig(level=logging.WARNING)

from prometheus.wp50_halt_problem import (
    run_halt_experiment,
    verify_wp50_exit_criteria,
    RecursionDepthSimulator,
    HaltOracle,
)
print("WP50 loaded ✓")

## Run the Halt Experiment

Runs CRLS at depths 1, 2, 3 (200 generations each).  This will take about 30 seconds.

In [ ]:
import time
t0 = time.time()
report = run_halt_experiment(n_generations=200, seed=42)
elapsed = time.time() - t0
print(f"Completed in {elapsed:.1f}s")
print()
print(report.summary())


## Accuracy Trajectories by Recursion Depth

Does deeper recursion lead to instability?

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use("Agg")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

depths   = sorted(report.depth_results)
colors   = {1: "blue", 2: "orange", 3: "red"}
n_panels = 2

if HAS_MPL:
    fig, axes = plt.subplots(n_panels, 1, figsize=(13, 9), sharex=False)

    # Panel 1: Accuracy traces for all depths
    ax = axes[0]
    for d in depths:
        dr   = report.depth_results[d]
        accs = dr.accuracies
        label = f"Depth {d}  (conv={dr.converged}, gain={dr.total_gain:+.3f})"
        ax.plot(accs, color=colors[d], label=label, linewidth=1.5, alpha=0.85)
        # Mark divergence events
        for ev in dr.divergence_events:
            ax.axvline(ev.generation, color=colors[d], linestyle=":", alpha=0.4, linewidth=1)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"CRLS Accuracy at Recursion Depths 1–3  |  Convergence boundary: depth {report.convergence_boundary}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 2: Rolling variance for all depths
    ax2 = axes[1]
    for d in depths:
        dr  = report.depth_results[d]
        var = dr.rolling_variance(window=20)
        ax2.plot(var, color=colors[d], label=f"Depth {d} variance", linewidth=1.5, alpha=0.85)
    # Divergence threshold
    # Approximate from first depth result (threshold is fixed)
    thr = 0.008
    ax2.axhline(thr, color="black", linestyle="--", label=f"Divergence threshold ({thr})")
    ax2.set_ylabel("Rolling Variance (window=20)")
    ax2.set_xlabel("Generation")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale("log")

    plt.tight_layout()
    plt.savefig("wp50_halt_experiment.png", dpi=100)
    plt.show()
    print("Plot saved → wp50_halt_experiment.png")
else:
    for d in depths:
        dr = report.depth_results[d]
        print(f"Depth {d}: converged={dr.converged}  final_acc={dr.final_accuracy:.4f}  "
              f"gain={dr.total_gain:+.4f}  reverts={dr.safety_reverts}  divs={len(dr.divergence_events)}")


## Hyperparameter Traces

How do `step_size` (depth-1) and `meta_lr` (depth-2) evolve?

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

    for d in depths:
        dr = report.depth_results[d]
        axes[0].plot(dr.step_size_trace, color=colors[d], label=f"Depth {d} step_size", linewidth=1.2)
        if any(x != dr.meta_lr_trace[0] for x in dr.meta_lr_trace):
            axes[1].plot(dr.meta_lr_trace, color=colors[d], label=f"Depth {d} meta_lr", linewidth=1.2)

    axes[0].set_ylabel("step_size (depth-1 param)")
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
    axes[1].set_ylabel("meta_lr (depth-2 param)")
    axes[1].set_xlabel("Generation")
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
    axes[0].set_title("Hyperparameter Evolution by Recursion Depth")
    plt.tight_layout()
    plt.savefig("wp50_hyperparams.png", dpi=100)
    plt.show()
    print("Plot saved → wp50_hyperparams.png")
else:
    print("Hyperparameter traces (first 10 entries per depth):")
    for d in depths:
        dr = report.depth_results[d]
        print(f"  Depth {d} step_size: {dr.step_size_trace[:10]}")


## Divergence Events

In [ ]:
import json
total_divs = sum(len(dr.divergence_events) for dr in report.depth_results.values())
print(f"Total divergence events: {total_divs}")
print()
for d in depths:
    dr = report.depth_results[d]
    if dr.divergence_events:
        print(f"Depth {d} divergence events:")
        for ev in dr.divergence_events:
            print(json.dumps(ev.to_dict(), indent=2))
        print()
    else:
        print(f"Depth {d}: no divergence events (safety gate not triggered)")
        print()


## Good's Verdict & Turing Statement

In [ ]:
print("Good's verdict:")
print(report.good_verdict)
print()
print("Turing statement:")
print(report.turing_statement)


## WP50 Exit Criteria

In [ ]:
criteria = verify_wp50_exit_criteria(report)
all_pass = all(criteria.values())
print(f"{'PASS' if all_pass else 'FAIL'} — WP50 Exit Criteria")
print()
for name, result in criteria.items():
    status = "✓" if result else "✗"
    print(f"  [{status}] {name}")
print()
print(f"All criteria pass: {all_pass}")


## Conclusion

WP50 makes Good's concern about the intelligence explosion **empirically falsifiable**:

1. **Depth 1** (standard meta-gradient): converges reliably.
2. **Depth 2** (meta-meta): step_size adapts to variance — usually stable, sometimes requires safety intervention.
3. **Depth 3** (meta-meta-meta): highest instability; safety gate fires most frequently.
4. **Convergence boundary**: the first depth where stability fails.  This is the empirical answer to "at what recursion depth does the halting problem bite?"

**Turing (1936)**: No general algorithm decides halting.  WP50 replaces the halting oracle with a **safety gate**: detect divergence, revert checkpoint, continue.  The bounded version of the halting problem is *empirically decidable*.

**Schmidhuber (2007)**: Gödel Machines require a formal proof that each self-modification improves expected utility.  WP50 provides the empirical analogue: the safety gate enforces convergence where formal proof is unavailable.
